# **Notebook for StitchingNet benchmark**
- Hyungjung Kim revises this notebook from the original Python code composed by Jingu Kang.

## **1. First stage: Wide-screening**

### Configuration

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import pandas as pd
import glob
import time

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

### 1-1. Data preprocessing with augmentation

In [ ]:
def create_dataframe(filepath):
    labels = [str(filepath[i]).split("/")[-2] for i in range(len(filepath))]
    filepath = pd.Series(filepath, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')
    df = pd.concat([filepath, labels], axis=1)
    # Reset index
    df = df.sample(frac=1, random_state=0).reset_index(drop=True)
    
    return df

def create_data_generators(img_dir, data_batch_size):
    '''
    img_dir : StitchingNet image path(Format: ././*/*/*.jpg)
    batch_size : Set the data batch size
    '''
    
    # Search the img_dir
    image_paths = glob.glob(img_dir)

    # Save dataframes
    df = create_dataframe(image_paths)

    # Split train, valid, and test as 6:2:2 ratio
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=0)
    # train_df.shape, test_df.shape
    
    train_df, valid_df = train_test_split(train_df, test_size=0.25, random_state=0)
    # train_df.shape, valid_df.shape


    # ImageDataGenerator for image augmentation
    train_datagen = ImageDataGenerator(rescale = 1./255,
    #                                   rotation_range=30, # 회전제한 각도 30도
    #                                   zoom_range=0.15, # 확대 축소 15%
    #                                   width_shift_range=0.2, # 좌우이동 20%
    #                                   height_shift_range=0.2, # 상하이동 20%
    #                                   shear_range=0.15, # 반시계방햐의 각도
    #                                   horizontal_flip=True, # 좌우 반전 True
                                    fill_mode="nearest")
    valid_datagen = ImageDataGenerator(rescale = 1./255,
    #                                   rotation_range=30, # 회전제한 각도 30도
    #                                   zoom_range=0.15, # 확대 축소 15%
    #                                   width_shift_range=0.2, # 좌우이동 20%
    #                                   height_shift_range=0.2, # 상하이동 20%
    #                                   shear_range=0.15, # 반시계방햐의 각도
    #                                   horizontal_flip=True, # 좌우 반전 True
                                    fill_mode="nearest")

    train_generator = train_datagen.flow_from_dataframe(train_df,
                                                        x_col='Filepath',
                                                        y_col='Label',
                                                        target_size=(224, 224),
                                                        batch_size=data_batch_size,
                                                        class_mode='categorical'
                                                        )
    validation_generator = valid_datagen.flow_from_dataframe(valid_df,
                                                        x_col='Filepath',
                                                        y_col='Label',
                                                        target_size=(224, 224),
                                                        batch_size=data_batch_size,
                                                        class_mode='categorical'
                                                        )
    # Test set으로 성능 확인
    test_datagen = ImageDataGenerator(rescale=1. / 255)
    test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    x_col='Filepath',
                                                    y_col='Label',
                                                    target_size=(224, 224),
                                                    batch_size=data_batch_size
                                                    )

    return train_generator, validation_generator, test_generator

In [ ]:
img_dir = "/kaggle/input/stitchingnet-dataset/*/*/*.jpg"
train, valid, test = create_data_generators(img_dir, 16)

### 1-2. Model building

In [ ]:
def build_model(model_name, trainable=False, num_classes=11):
    '''
    model_name: Model name to use
    trainable: Fine-tuning condition; False fixed
    num_classes: Number of classes; 11 fixed
    '''
    
    cnn_models = ['VGG19','DenseNet121','DenseNet169','DenseNet201','EfficientNetB0','EfficientNetB1','EfficientNetB7','EfficientNetV2B0','EfficientNetV2B3','EfficientNetV2L','EfficientNetV2M','EfficientNetV2S','InceptionV3','MobileNet','MobileNetV2','MobileNetV3Large','MobileNetV3Small','ResNet101','ResNet101V2','ResNet152','ResNet152V2','ResNet50','ResNet50V2','Xception']

    # For CNN models
    if model_name in cnn_models:

        # Model initilization, based on ImangeNet
        m_layer = getattr(tf.keras.applications, model_name)(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        m_layer.trainable = trainable

        x = m_layer.output
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(1024, activation='relu')(x)  # Add a dense layer
        output_layer = tf.keras.layers.Dense(11, activation='softmax')(x)

        model = tf.keras.models.Model(inputs = m_layer.input, outputs = output_layer)

    # For Vit models
    else:
        m_name = model_name.lower()
        vit_url = f"https://www.kaggle.com/models/spsayakpaul/vision-transformer/TensorFlow2/{m_name}/1"
        
        fe_layer = hub.KerasLayer(vit_url, trainable=trainable)
        
        inputs = tf.keras.Input(shape=(224,224,3))
        fe_output = fe_layer(inputs)
        x = tf.keras.layers.Dense(1024, activation='relu')(fe_output)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs, name=model_name)

    return model

### 1-3. Model evaluation

In [ ]:
def mode_evaluate(model,test):
    test_loss, test_accuracy = model.evaluate(test)

    all_pre = []
    all_label = []
    all_time = 0

    for i in range(len(test)):
        batch_x, batch_y = test[i]
        start_time = time.time()
        y_pred = model.predict(batch_x)
        end_time = time.time()
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_label = np.argmax(batch_y, axis=1)

        all_pre.extend(y_pred_classes)
        all_label.extend(y_label)
        all_time += end_time - start_time

    # Calculate Precision, Recall, and F1-Score
    precision = precision_score(all_label, all_pre, average='weighted', zero_division=0)
    recall = recall_score(all_label, all_pre, average='weighted', zero_division=0)
    f1 = f1_score(all_label, all_pre, average='weighted', zero_division=0)
    # print(all_label, len(all_label))
    # print(all_pre, len(all_pre))
    # print(cnt)
    
    # Calculate Inference Time
    inference_time = all_time/len(test)

    return test_accuracy, test_loss, precision, recall, f1, inference_time

### 1-4. Run benchmarks

In [ ]:
# model_list = ['DenseNet121','DenseNet169','DenseNet201','EfficientNetB0','EfficientNetB1','EfficientNetB7','EfficientNetV2B0','EfficientNetV2B3','EfficientNetV2L','EfficientNetV2M','EfficientNetV2S','InceptionV3', 'MobileNet','MobileNetV2','MobileNetV3Large','MobileNetV3Small','ResNet101','ResNet101V2','ResNet152','ResNet152V2','ResNet50','ResNet50V2','Xception',              'vit-b16-fe','vit-r26-s32-medaug-fe', 'vit-r50-l32-fe','vit-b32-fe','vit-b8-fe', 'vit-l16-fe','vit-r26-s32-lightaug-fe','vit-s16-fe','VGG19']

model_list = ['VGG19', 'ResNet50']

model_result = []

for model_name in model_list:
    try:
        model = build_model(model_name)
    except:
        print("Failed to build %s", model_name)
        continue

    # model.summary()

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    epochs = 100

    print("Model training started: ", model_name)
    
    model.fit(train, validation_data=valid, epochs=epochs, callbacks=[early_stopping])
    
    acc, test_loss, pre, rec, f1, inf_time = mode_evaluate(model, test)

    # Evaluation results
    print(f"Test accuracy: {acc:.5f}")
    print(f"Test loss: {test_loss:.5f}")
    print(f"Precision: {pre:.5f}")
    print(f"Recall: {rec:.5f}")
    print(f"F1-score: {f1:.5f}")
    print(f"Inference time: {inf_time:.5f} seconds")

    model_result.append([model_name, acc, pre, rec, f1, inf_time])

results_df = pd.DataFrame(model_result, columns=['Model','Accuracy','Precision','Recall','F1-score','Inference time'])
results_df.to_csv('./model_flatten_vgg19.csv', index=False)